# Решения: практикум CI/correlation

**Для преподавателя.** Эталон к `lesson.ipynb` и `homework.ipynb`. Не показывать ученикам до сдачи.

## Карта эталона

**Фокус:** Устойчивость вывода по сегментам.

Сравниваем источники трафика, устройства и скидки. Малый сегмент даёт широкий интервал, поэтому знак точечной оценки нельзя превращать в уверенный product-вывод.

Эталон разделён на исполняемые секции в том же порядке, что `lesson.ipynb` и `homework.ipynb`. После каждой секции сверяйте не только значение, но и способ вычисления.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd


def _find(name):
    for path in (
        Path(name),
        Path("../") / name,
        Path("../../data") / name,
        Path("../data") / name,
        Path("../../../data") / name,
    ):
        if path.exists():
            return path.resolve()
    return "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/modules/08_06_ab_startup/data/" + name


CSV_PATH = _find('startup_ab.csv')
df = pd.read_csv(CSV_PATH)
df['variant_b'] = (df['variant'] == 'B').astype(int)


## 0.1. Предзаданные сегменты

Перечислите сегменты до просмотра uplift. Это защищает практику от post-hoc выбора только тех групп, где случайно получился удобный знак.

**Эталон.** Эта ячейка фиксирует тот же контракт, который ученик заполняет до основного расчёта.

In [ ]:
planned_segments = ('traffic_source', 'device')
assert all(col in df.columns for col in planned_segments)

## 0.2. Размеры групп и баланс вариантов

Соберите таблицу `segment_n` по source, device и variant. До CI убедитесь, что каждая запланированная ячейка содержит наблюдения обеих групп.

**Эталон.** Эта ячейка фиксирует тот же контракт, который ученик заполняет до основного расчёта.

In [ ]:
segment_n = (
    df.groupby(['traffic_source', 'device', 'variant'])
    .size().rename('n').reset_index()
)
assert int(segment_n['n'].sum()) == len(df)

## Решение 1

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
seg = (
    df.groupby(['traffic_source', 'device'])
    .agg(n=('user_id', 'count'), conv=('converted', 'mean'))
    .reset_index()
)

def ci_uplift(part, seed=0, n_iter=1500):
    rng = np.random.default_rng(seed)
    a = part[part['variant'] == 'A']['converted'].to_numpy()
    b = part[part['variant'] == 'B']['converted'].to_numpy()
    if len(a) < 20 or len(b) < 20:
        return np.nan, np.nan
    vals = []
    for _ in range(n_iter):
        vals.append(float(rng.choice(b, len(b), replace=True).mean() - rng.choice(a, len(a), replace=True).mean()))
    return tuple(np.quantile(vals, [0.025, 0.975]))

## Решение 2

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
rows = []

for i, src in enumerate(sorted(df['traffic_source'].unique())):
    part = df[df['traffic_source'] == src]
    low, high = ci_uplift(part, seed=390 + i)
    rows.append({'traffic_source': src, 'ci_low': low, 'ci_high': high})

## Решение 3

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
ci_source = pd.DataFrame(rows)

SIGN_NOTE = (
    'В отдельных сегментах знак может отличаться из-за шума и разных размеров подвыборок. '
    'Поэтому важны интервалы и проверка устойчивости эффекта.'
)

corr_age = float(df['age'].corr(df['converted']))

## Решение 4

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
REPORT_LINE = (
    'По сегментам видно, что эффект B не обязан быть одинаковым во всех каналах: '
    'часть различий может быть статистическим шумом в малых группах.'
)

rows2 = []

for d, part in df.groupby('discount_pct'):
    low, high = ci_uplift(part, seed=410 + int(d))
    rows2.append({'discount_pct': int(d), 'ci_low': low, 'ci_high': high})

## Решение 5

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
ci_discount = pd.DataFrame(rows2).sort_values('discount_pct')

corr_prior = float(df['prior_visits_30d'].corr(df['converted']))

STABILITY_NOTE = (
    'Эффект устойчивее, когда знак uplift совпадает в ключевых сегментах и интервалы не слишком широкие. '
    'Если интервалы широкие, нужен больший объём данных.'
)

## Решение 6

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
CHECKLIST = (
    'Перед product-решением проверяем: протокол эксперимента, размер выборки, CI эффекта, '
    'чувствительность к сегментам, отсутствие peeking и согласованность с бизнес-ограничениями.'
)

print(seg.head())

print(ci_source)

## Решение 7

Выполните секцию после всех предыдущих: переменные намеренно переиспользуются, чтобы эталон воспроизводил полный аналитический pipeline.

In [ ]:
print(corr_age, corr_prior)

## Проверка преподавателя

Запустите `Run All`. Эталон должен завершиться без исключений; итоговые числа должны совпадать при повторном запуске благодаря фиксированным seed. Текстовый вывод проверяется на согласованность с направлением uplift, p-value и границами CI.